# Lab Assignment - 2
## Cloud Computing Lab — Matrix Operations using Python

**Tasks:**
1. Create a 20000 x 10000 data file using Python random module
2. Assign subsequent rows to matrix first row elements
3. Group similar elements into matrix
4. Add and subtract matrices
5. Create n x n square matrix where all sub-matrices have even sum of opposite corner elements
6. Row-wise element addition in tuple matrix

---
## Task 1: Create a 20000 × 10000 Data File Using Python Random Module

**Objective:** Generate a large data file with 20,000 rows and 10,000 columns using various random generator parameters. This file serves as a database for all subsequent matrix operations.

**Note:** Creating the full 20000×10000 file uses ~1.5 GB of disk. We generate it in chunks to be memory-efficient, then demonstrate operations on a sampled subset.

In [ ]:
# Task 1: Create 20000 x 10000 data file using Python random module
import random
import numpy as np
import os
import time

ROWS = 20000
COLS = 10000
FILENAME = 'data_matrix_20000x10000.csv'

print(f"Creating {ROWS} x {COLS} data file...")
print("Using multiple random generator parameters:")
print("  - random.randint   : for integer values")
print("  - random.uniform   : for float values")
print("  - random.gauss     : for Gaussian distribution")
print()

# Random generator functions to mix different distributions
generators = [
    lambda: random.randint(0, 100),           # Integers 0-100
    lambda: round(random.uniform(0.0, 1.0), 4),  # Floats 0.0-1.0
    lambda: round(random.gauss(50, 15), 4),   # Gaussian mean=50, std=15
]

random.seed(42)  # For reproducibility
start = time.time()

# Write in chunks to avoid memory overflow
CHUNK_SIZE = 500  # rows per chunk
with open(FILENAME, 'w') as f:
    for chunk_start in range(0, ROWS, CHUNK_SIZE):
        chunk_end = min(chunk_start + CHUNK_SIZE, ROWS)
        lines = []
        for row in range(chunk_end - chunk_start):
            # Use different generators for different column segments
            gen_func = generators[(chunk_start + row) % len(generators)]
            values = [str(gen_func()) for _ in range(COLS)]
            lines.append(','.join(values))
        f.write('\n'.join(lines) + '\n')
        if (chunk_start // CHUNK_SIZE) % 10 == 0:
            pct = (chunk_start / ROWS) * 100
            print(f"  Progress: {chunk_start}/{ROWS} rows ({pct:.0f}%)...")

elapsed = time.time() - start
file_size_mb = os.path.getsize(FILENAME) / (1024 * 1024)

print(f"\n✅ File '{FILENAME}' created successfully!")
print(f"   Rows    : {ROWS}")
print(f"   Columns : {COLS}")
print(f"   Size    : {file_size_mb:.2f} MB")
print(f"   Time    : {elapsed:.2f} seconds")

In [ ]:
# Load a sample (100 x 20) for demonstration of operations
# Reading full 20000x10000 into RAM would require ~1.5GB
print("Loading sample (100 rows x 20 columns) for matrix operations...")

sample_rows = []
with open(FILENAME, 'r') as f:
    for i, line in enumerate(f):
        if i >= 100:
            break
        row = list(map(float, line.strip().split(',')[:20]))
        sample_rows.append(row)

sample_matrix = np.array(sample_rows)
print(f"Sample matrix shape: {sample_matrix.shape}")
print(f"First 3 rows, first 5 cols:\n{sample_matrix[:3, :5]}")

---
## Task 2: Assign Subsequent Rows to Matrix First Row Elements

**Objective:** Use the first row of the matrix as a reference/key. For each element in the first row, assign the corresponding column's subsequent rows to it — effectively reorganizing the matrix so that first-row elements serve as group labels/keys, and the data below each key is collected under it.

In [ ]:
# Task 2: Assign subsequent rows to matrix first row elements
import numpy as np

print("=" * 60)
print("TASK 2: Subsequent Rows Assigned to First Row Elements")
print("=" * 60)

# Create a small example matrix for clear demonstration
np.random.seed(0)
matrix = np.array([
    [10, 20, 30, 40],    # Row 0 — First row: keys/labels
    [ 1,  2,  3,  4],    # Row 1
    [ 5,  6,  7,  8],    # Row 2
    [ 9, 11, 13, 15],    # Row 3
    [16, 17, 18, 19],    # Row 4
])

print("\nOriginal Matrix (5 x 4):")
print(matrix)

# First row elements act as keys
first_row = matrix[0]          # [10, 20, 30, 40]
subsequent_rows = matrix[1:]   # Rows 1 onwards

print(f"\nFirst Row (Keys): {first_row}")
print(f"Subsequent Rows (Values):\n{subsequent_rows}")

# Assign subsequent rows to each first-row element (column-wise grouping)
assigned = {}
for col_idx, key in enumerate(first_row):
    assigned[key] = subsequent_rows[:, col_idx]  # all subsequent values in that column

print("\n📋 Assignment: First Row Element → Subsequent Column Values")
print("-" * 45)
for key, values in assigned.items():
    print(f"  Key [{key:2d}] → {values}")

# Also demonstrate with the actual data file (first 6 rows, 5 cols)
print("\n" + "=" * 60)
print("Demonstration on ACTUAL DATA FILE (6 rows x 5 cols):")
print("=" * 60)
sub = sample_matrix[:6, :5]
print("Matrix subset:\n", sub)

first_row_actual = sub[0]
subsequent_actual = sub[1:]
print(f"\nFirst row keys: {first_row_actual}")
assigned_actual = {}
for col_idx, key in enumerate(first_row_actual):
    assigned_actual[round(key, 2)] = subsequent_actual[:, col_idx]

print("\nAssignment (First Row Key → Subsequent Column Data):")
for key, vals in assigned_actual.items():
    print(f"  Key [{key}] → {np.round(vals, 2)}")

---
## Task 3: Group Similar Elements into Matrix

**Objective:** Scan through the data file/matrix and group elements that are similar (i.e., fall within the same value range/bucket) together into a new matrix structure. Elements are grouped by their integer bucket (floor division).

In [ ]:
# Task 3: Group similar elements into matrix
import numpy as np
from collections import defaultdict

print("=" * 60)
print("TASK 3: Group Similar Elements into Matrix")
print("=" * 60)

# Create a sample matrix with repeated/similar values
np.random.seed(5)
data = np.random.randint(1, 51, size=(6, 6))
print("\nOriginal Matrix (6 x 6):")
print(data)

# Define bucket size for grouping
BUCKET_SIZE = 10  # Group: 1-10, 11-20, 21-30, 31-40, 41-50

# Group all elements by their bucket
groups = defaultdict(list)
for r in range(data.shape[0]):
    for c in range(data.shape[1]):
        val = data[r, c]
        bucket = ((val - 1) // BUCKET_SIZE) * BUCKET_SIZE + 1
        groups[bucket].append((val, r, c))  # Store value and position

print(f"\n📊 Grouping elements into buckets of size {BUCKET_SIZE}:")
print("-" * 50)
for bucket in sorted(groups.keys()):
    values = [x[0] for x in groups[bucket]]
    positions = [(x[1], x[2]) for x in groups[bucket]]
    print(f"  Bucket [{bucket:2d}–{bucket+BUCKET_SIZE-1:2d}]: {sorted(values)}")
    print(f"             Positions: {positions}")

# Reconstruct a grouped matrix — rows = buckets, cols = elements in that bucket
print("\n📦 Grouped Matrix (rows = group buckets, padded with -1):")
max_count = max(len(v) for v in groups.values())
bucket_keys = sorted(groups.keys())
grouped_matrix = np.full((len(bucket_keys), max_count), -1)  # -1 = empty

for i, bucket in enumerate(bucket_keys):
    vals = sorted([x[0] for x in groups[bucket]])
    grouped_matrix[i, :len(vals)] = vals

print("  (Rows: groups in order. -1 = no element in that slot)")
print(grouped_matrix)

# Also apply to actual data file sample
print("\n" + "=" * 60)
print("Grouping on ACTUAL DATA FILE (using integer values 0-100):")
print("=" * 60)
flat_vals = sample_matrix[:10, :10].flatten().astype(int)
groups_actual = defaultdict(list)
for v in flat_vals:
    b = (v // 10) * 10
    groups_actual[b].append(v)

print("Bucket → Elements:")
for b in sorted(groups_actual):
    print(f"  [{b:3d}–{b+9:3d}]: {sorted(groups_actual[b])}")

---
## Task 4: Add and Subtract Matrices in Python

**Objective:** Perform element-wise matrix addition and subtraction using Python, both with and without NumPy, to demonstrate different approaches.

In [ ]:
# Task 4: Add and subtract matrices in Python
import numpy as np
import random

print("=" * 60)
print("TASK 4: Matrix Addition and Subtraction")
print("=" * 60)

# ── Method 1: Pure Python (nested lists) ──────────────────────────
print("\n▶ Method 1: Using Pure Python (Nested Lists)")
print("-" * 40)

def matrix_add(A, B):
    """Element-wise addition of two matrices (pure Python)."""
    rows = len(A)
    cols = len(A[0])
    result = [[A[i][j] + B[i][j] for j in range(cols)] for i in range(rows)]
    return result

def matrix_subtract(A, B):
    """Element-wise subtraction of two matrices (pure Python)."""
    rows = len(A)
    cols = len(A[0])
    result = [[A[i][j] - B[i][j] for j in range(cols)] for i in range(rows)]
    return result

def print_matrix(M, name):
    print(f"  {name}:")
    for row in M:
        print("   ", row)

random.seed(10)
A_list = [[random.randint(1, 20) for _ in range(4)] for _ in range(3)]
B_list = [[random.randint(1, 20) for _ in range(4)] for _ in range(3)]

print_matrix(A_list, "Matrix A")
print_matrix(B_list, "Matrix B")
print_matrix(matrix_add(A_list, B_list),      "A + B (Pure Python)")
print_matrix(matrix_subtract(A_list, B_list), "A - B (Pure Python)")

# ── Method 2: NumPy ───────────────────────────────────────────────
print("\n▶ Method 2: Using NumPy")
print("-" * 40)

np.random.seed(10)
A_np = np.random.randint(1, 50, size=(4, 4))
B_np = np.random.randint(1, 50, size=(4, 4))

print(f"  Matrix A (4x4):\n{A_np}")
print(f"\n  Matrix B (4x4):\n{B_np}")
print(f"\n  A + B:\n{A_np + B_np}")
print(f"\n  A - B:\n{A_np - B_np}")
print(f"\n  Sum of all elements in (A+B): {(A_np + B_np).sum()}")
print(f"  Sum of all elements in (A-B): {(A_np - B_np).sum()}")

# ── Method 3: On actual data file slices ─────────────────────────
print("\n▶ Method 3: On Actual Data File (20000x10000) — Slices")
print("-" * 40)

# Use two non-overlapping slices of our data file as matrices
rows_loaded = []
with open(FILENAME, 'r') as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        rows_loaded.append(list(map(float, line.strip().split(',')[:5])))

rows_loaded2 = []
with open(FILENAME, 'r') as f:
    for i, line in enumerate(f):
        if i < 10:
            continue
        if i >= 20:
            break
        rows_loaded2.append(list(map(float, line.strip().split(',')[:5])))

M1 = np.array(rows_loaded)
M2 = np.array(rows_loaded2)
print(f"  Slice 1 (rows 0-9, cols 0-4):\n{np.round(M1, 2)}")
print(f"\n  Slice 2 (rows 10-19, cols 0-4):\n{np.round(M2, 2)}")
print(f"\n  M1 + M2:\n{np.round(M1 + M2, 2)}")
print(f"\n  M1 - M2:\n{np.round(M1 - M2, 2)}")

---
## Task 5: Create n × n Square Matrix Where All Sub-Matrices Have Even Sum of Opposite Corner Elements

**Objective:** Construct an n × n matrix such that for every sub-matrix (of any size), the sum of its two pairs of opposite corner elements is always **even**.

**Mathematical Insight:**
> The sum of two integers is even **if and only if** both integers share the same parity (both even, or both odd).
>
> **Simplest valid construction:** Fill the entire n × n matrix with **even numbers**.
> Since **even + even = even**, every possible opposite-corner sum across every sub-matrix will automatically be even — with zero exceptions.

This is verified exhaustively in the code below for n = 4, 6, and 8.

In [ ]:
# Task 5: n x n square matrix where all sub-matrices have even sum of opposite corner elements
import numpy as np
import random

print("=" * 60)
print("TASK 5: n×n Matrix — Even Sum of Opposite Corner Elements")
print("=" * 60)

print("""
Mathematical Foundation:
────────────────────────
The sum of two integers is EVEN if and only if both integers
have the SAME parity (both even OR both odd).

For a sub-matrix with corners at (r1,c1), (r1,c2), (r2,c1), (r2,c2):
  • Opposite pair 1: M[r1,c1] + M[r2,c2]  must be even
  • Opposite pair 2: M[r1,c2] + M[r2,c1]  must be even

SOLUTION: Fill the entire n×n matrix with EVEN numbers.
  Even + Even = EVEN — always.  No constraint on positions.
  This guarantees the condition holds for ALL sub-matrices.
""")

def build_even_matrix(n, seed=42):
    """
    Build an n×n matrix filled entirely with distinct even numbers.
    Since even + even = even, every sub-matrix's opposite corner
    element sums will always be even.
    """
    random.seed(seed)
    even_pool = list(range(2, n * n * 4, 2))  # large pool of evens
    random.shuffle(even_pool)
    selected = even_pool[:n * n]
    matrix = np.array(selected, dtype=int).reshape(n, n)
    return matrix

def verify_all_submatrices(matrix):
    """Exhaustively verify: for every sub-matrix, both pairs of
    opposite corners sum to an even number."""
    n = matrix.shape[0]
    violations = 0
    total_checks = 0
    for r1 in range(n):
        for r2 in range(r1 + 1, n):
            for c1 in range(n):
                for c2 in range(c1 + 1, n):
                    # Pair 1: top-left + bottom-right
                    s1 = int(matrix[r1, c1]) + int(matrix[r2, c2])
                    # Pair 2: top-right + bottom-left
                    s2 = int(matrix[r1, c2]) + int(matrix[r2, c1])
                    total_checks += 2
                    if s1 % 2 != 0:
                        violations += 1
                    if s2 % 2 != 0:
                        violations += 1
    return total_checks, violations

# Build and test for multiple values of n
for n in [4, 6, 8]:
    M = build_even_matrix(n)
    checks, violations = verify_all_submatrices(M)

    print(f"{'─' * 55}")
    print(f"  n = {n}  →  {n}×{n} Matrix (all even elements):")
    print(M)
    print(f"  Total sub-matrix corner checks : {checks}")
    print(f"  Violations (odd sums)          : {violations}")
    status = "✅ ALL sub-matrices satisfy the condition!" if violations == 0 else f"❌ {violations} violations"
    print(f"  {status}")

# Detailed walkthrough for n=4
print("\n" + "=" * 55)
print("Detailed Walkthrough — n = 4:")
n = 4
M = build_even_matrix(n)
print(M)

r1, r2, c1, c2 = 0, 3, 0, 3  # outermost corners
print(f"\nExample Sub-matrix: full 4×4 (rows 0-3, cols 0-3)")
print(f"  Top-Left     M[{r1},{c1}] = {M[r1,c1]}")
print(f"  Bottom-Right M[{r2},{c2}] = {M[r2,c2]}")
print(f"  Sum (opposite pair 1) = {M[r1,c1]} + {M[r2,c2]} = {M[r1,c1]+M[r2,c2]} → {'EVEN ✅' if (M[r1,c1]+M[r2,c2])%2==0 else 'ODD ❌'}")
print(f"  Top-Right    M[{r1},{c2}] = {M[r1,c2]}")
print(f"  Bottom-Left  M[{r2},{c1}] = {M[r2,c1]}")
print(f"  Sum (opposite pair 2) = {M[r1,c2]} + {M[r2,c1]} = {M[r1,c2]+M[r2,c1]} → {'EVEN ✅' if (M[r1,c2]+M[r2,c1])%2==0 else 'ODD ❌'}")

r1, r2, c1, c2 = 1, 3, 0, 2
print(f"\nExample Sub-matrix: rows 1-3, cols 0-2")
sub = M[r1:r2+1, c1:c2+1]
print(sub)
print(f"  Top-Left  + Bottom-Right = {M[r1,c1]} + {M[r2,c2]} = {M[r1,c1]+M[r2,c2]} → {'EVEN ✅' if (M[r1,c1]+M[r2,c2])%2==0 else 'ODD ❌'}")
print(f"  Top-Right + Bottom-Left  = {M[r1,c2]} + {M[r2,c1]} = {M[r1,c2]+M[r2,c1]} → {'EVEN ✅' if (M[r1,c2]+M[r2,c1])%2==0 else 'ODD ❌'}")

---
## Task 6: Row-Wise Element Addition in Tuple Matrix

**Objective:** Given a matrix where each element is a tuple (or each row contains tuples), perform row-wise element addition — i.e., add corresponding elements across tuples in each row.

In [ ]:
# Task 6: Row-wise element addition in tuple matrix
import numpy as np
from functools import reduce

print("=" * 60)
print("TASK 6: Row-Wise Element Addition in Tuple Matrix")
print("=" * 60)

# ── Representation 1: Matrix of tuples (each cell is a tuple) ────
print("\n▶ Representation 1: Matrix where each element is a tuple")
print("-" * 50)

# Each element is a 3-tuple
tuple_matrix = [
    [(1, 2, 3),  (4, 5, 6),  (7, 8, 9)   ],
    [(10, 11, 12),(13, 14, 15),(16, 17, 18)],
    [(2, 4, 6),  (1, 3, 5),  (7, 9, 11)  ],
    [(0, 1, 0),  (2, 3, 4),  (5, 6, 7)   ],
]

print("Tuple Matrix (4 rows × 3 cols, each cell = 3-tuple):")
for i, row in enumerate(tuple_matrix):
    print(f"  Row {i}: {row}")

def row_wise_tuple_addition(matrix):
    """
    For each row: sum corresponding positions across all tuples.
    E.g. row = [(1,2,3), (4,5,6), (7,8,9)]
    → result = (1+4+7, 2+5+8, 3+6+9) = (12, 15, 18)
    """
    results = []
    for row in matrix:
        # zip unpacks positional elements across all tuples
        row_sum = tuple(sum(pos_vals) for pos_vals in zip(*row))
        results.append(row_sum)
    return results

row_sums = row_wise_tuple_addition(tuple_matrix)
print("\nRow-wise element addition result:")
print("  (Each result tuple = sum of corresponding positions across all tuples in that row)")
for i, (orig_row, result) in enumerate(zip(tuple_matrix, row_sums)):
    print(f"  Row {i}: {orig_row}")
    print(f"      →   Sum = {result}")

# ── Representation 2: Rows of numeric tuples (each row is a tuple) ─
print("\n▶ Representation 2: Each row is itself a tuple of numbers")
print("-" * 50)

row_tuple_matrix = (
    (5, 10, 15, 20, 25),
    (3,  6,  9, 12, 15),
    (1,  2,  3,  4,  5),
    (8,  7,  6,  5,  4),
    (2,  4,  6,  8, 10),
)

print("Tuple Matrix:")
for i, row in enumerate(row_tuple_matrix):
    print(f"  Row {i}: {row}")

# Row-wise sum (sum each row's elements)
print("\nRow-wise element addition (sum of each row):")
row_totals = []
for i, row in enumerate(row_tuple_matrix):
    total = sum(row)
    row_totals.append(total)
    print(f"  Row {i}: {row}  →  Sum = {total}")

print(f"\n  Row sums: {row_totals}")
print(f"  Column-wise sum across all rows: {tuple(sum(col) for col in zip(*row_tuple_matrix))}")

# ── Representation 3: NumPy structured approach ───────────────────
print("\n▶ Representation 3: NumPy row-wise addition")
print("-" * 50)

np.random.seed(7)
# 5 rows, each row has 4 tuples of length 3 → shape (5, 4, 3)
np_tuple_matrix = np.random.randint(1, 15, size=(5, 4, 3))
print("NumPy tuple matrix shape:", np_tuple_matrix.shape)
print("(5 rows, each row has 4 tuples of 3 elements)\n")

for i in range(np_tuple_matrix.shape[0]):
    row = np_tuple_matrix[i]  # shape (4, 3)
    row_sum = np.sum(row, axis=0)  # sum across the 4 tuples → shape (3,)
    print(f"  Row {i}: {row.tolist()}")
    print(f"      →  Element-wise sum = {row_sum.tolist()}")

print("\n" + "=" * 60)
print("Lab Assignment 2 — ALL TASKS COMPLETE ✅")
print("=" * 60)